In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer


I0000 00:00:1789161384.401555  278870 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789161384.402145  278870 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789161384.441876  278870 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789161385.413858  278870 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
tokenizer = Tokenizer()

In [3]:
data = """In the village by the river lived a man named Old Tom Kane
He worked the land from dawn to dusk through sunshine and through rain
His father left him nothing but a horse and rusty plow
But Tom he built a farmstead that would make the whole town bow

He threw a grand party for his neighbors and his kin
They came from every corner just to hear the music spin
And if you care to listen I will tell you what befell
The dancing and the laughter at the Old Tom Kane farewell

Myself I got an invitation folded in an envelope
For all the local children and the young folks full of hope
And in a single moment friends and strangers old and new
Were spinning round the barnyard like the morning drops of dew

Mary Clare the miller's daughter gave a smile so bright and true
She waved to me from yonder and I waved right back on cue
And soon I met her brother who had come from cross the sea
Just in time to join the dancing at the Old Tom Kane spree

The fiddler played till midnight and the moon rose high and clear
The children fell asleep at last still clutching cider near
And Old Tom Kane stood watching with a twinkle in his eye
Knowing all his work had built a joy that wouldn't die"""

corpus = data.lower().split("\n")
# separa o texto em linhas e coloca em minusculo

In [5]:
tokenizer.fit_on_texts(corpus)
# gera um dicionario de palavras com a frequencia de cada palavra no texto
total_words = len(tokenizer.word_index) + 1
# total de palavras no dicionario + 1 para o padding

In [6]:
input_sequences = []

for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    # converte o texto em uma sequencia de numeros
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)
        # gera as sequencias de n-gramas
        # ele gera sub-listas para cada palavra, podendo dar sequencia prevendo a proxima palavra

In [7]:
max_sequence_len = max([len(x) for x in input_sequences])
# pega a maior frase para definir o tamanho do padding

In [8]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [9]:
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))
# adiciona 0 com o padding para cada linha da lista que nao tiver o mesmo que a maior linha

In [10]:
xs = input_sequences[:,:-1]
# separa as sequencias de n-gramas em features e labels
labels = input_sequences[:,-1]

In [11]:
ys = tf.keras.utils.to_categorical(labels, num_classes=total_words)
# transforma os labels em one hot encoder, exemplo [0,0,0,0,0,1,0,0,0,0]

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.optimizers import Adam

In [13]:
model = Sequential()
model.add(Embedding(total_words, 240, input_length=max_sequence_len-1))
# cria um vetor de embedding para cada palavra, com 240 dimensões. CAda vetor contem pesos que a palavra representa
model.add(Bidirectional(LSTM(150)))
# garante que a rede nao perca a memoria das primeiras palavras
model.add(Dense(total_words, activation='softmax'))
# camada de saida com o total de palavras de dicionario
adam = Adam(learning_rate=0.01)# otimizador Adam com taxa de aprendizado de 0.01
model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
# parametros de compilação do modelo 
history = model.fit(xs, ys, epochs=100, verbose=1)

Epoch 1/100


/home/beuren/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
E0000 00:00:1789161395.998921  278870 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.0374 - loss: 5.0168 
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0981 - loss: 4.7193
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.1355 - loss: 4.0519
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.1869 - loss: 3.2017
Epoch 5/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.2664 - loss: 2.4382
Epoch 6/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.4626 - loss: 1.7422
Epoch 7/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6308 - loss: 1.2016
Epoch 8/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7710 - loss: 0.7039
Epoch 9/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8645 - loss: 0.4599
Epoch 10/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9252 - loss: 0.2863
Epoch 11/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9579 - loss: 0.1878
Epoch 12/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9579 - loss: 0.1455


In [14]:
seed_text = "I made a poetry machine"
# frase inical para gerar a proxima palavra
next_words = 100
# numero de apalavras que serao geradas

#  esse loop é que vai fazer a previsão da proxima palavra e vai concatenando a frase com a palavra prevista
for _ in range(next_words):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = np.argmax(model.predict(token_list), axis=-1)
    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted:
            output_word = word
            break
    seed_text += " " + output_word

print(seed_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━